[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NASA-Gradient-Descent-Farmers/NASA-warm-up/blob/sant-calibration/Notebooks/yield_planning.ipynb)

# All of Mexico, 46 seasons, and a pre-season planning model

* **Author:** Santiago (Gradient Descent Farmers) · **Branch:** `sant-calibration`
* **Follows:** `yield_panel.ipynb` (the panel and the bar) and `yield_panel_sources.ipynb` (more NASA sources, the SIAP district finding), both on central Mexico.
* **Code:** `src/yield_panel/historic.py` (pre-2003 SIAP), `planning.py` (pre-season features), `plan_build.py` (panels, evaluation, outlook), `share.py` (data package).

Three questions:
1. **Do the in-season results hold for the whole country?**
2. **Does a longer history help?** SIAP's district data reach back to 1999 and state data to 1980, from the archived SAGARPA files (mirrored on datamx.io).
3. **Can farmers plan ahead?** A model that only uses what is known by April 30: ocean indices (ENSO, tropical Atlantic, PDO), pre-season rain and temperature, last season's weather and outcome.

Every model is still scored against the same bar (each unit's own average + trend), one year ahead, with year-level bootstrap intervals.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/NASA-Gradient-Descent-Farmers/NASA-warm-up.git"
BRANCH = "sant-calibration"
if "google.colab" in sys.modules:
    if not Path("/content/NASA-warm-up").exists():
        !git clone -q --branch {BRANCH} {REPO_URL} /content/NASA-warm-up
    !pip install -q "earthaccess>=0.11" pyhdf pyarrow
    ROOT = Path("/content/NASA-warm-up")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "yield_panel").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.yield_panel import historic, planning, plan_build

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
OUT = Path("data/yield_panel/planning")
NAT = Path("data/yield_panel/ablation_national")

### Getting the data

The fastest route is the **shared data package** (`data/shared/*.parquet`, made by `python -m src.yield_panel.share export`): unzip it into `data/shared/` and the cell below rebuilds the working files from it. Without it, the pipeline re-downloads everything (several hours on a laptop; see `src/yield_panel/README.md`).

In [ ]:
from src.yield_panel import share
if Path("data/shared/manifest.json").exists() and not (OUT / "district.parquet").exists():
    OUT.mkdir(parents=True, exist_ok=True)
    for name, target in [("planning_municipal", OUT / "municipal.parquet"), ("planning_district", OUT / "district.parquet"),
                         ("planning_state", OUT / "state.parquet")]:
        share.load(name).to_parquet(target, index=False)
print("planning panels:", sorted(p.name for p in OUT.glob("*.parquet")))

## 1. Forty-six seasons of rainfed maize

In [ ]:
states = pd.read_parquet(OUT / "state.parquet")
nat = states[states.crop == "maize"].groupby("year").agg(sown=("sown_ha", "sum"), lost=("lost_ha", "sum"),
                                                        harvested=("harvested_ha", "sum"), prod=("production_t", "sum"),
                                                        oni_fma=("oni_fma", "first"))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].plot(nat.index, nat.sown / 1e6, "o-", ms=3); axes[0].set(title="Rainfed spring-summer maize sown", ylabel="million ha")
axes[1].plot(nat.index, nat["prod"] / nat.harvested, "o-", ms=3); axes[1].set(title="Yield on harvested area", ylabel="t/ha")
axes[2].bar(nat.index, nat.lost / nat.sown, color=np.where(nat.oni_fma < -0.5, "tab:blue", np.where(nat.oni_fma > 0.5, "tab:red", "tab:gray")))
axes[2].set(title="Share of sown area lost (blue: La Nina spring, red: El Nino spring)", ylabel="fraction")
for ax in axes: ax.axvline(1998.5, color="k", lw=0.5, ls=":"); ax.axvline(2002.5, color="k", lw=0.5, ls=":")
fig.tight_layout(); plt.show()
print("Dotted lines: state-level data before 1999, district-level 1999-2002, municipal from 2003.")

## 2. In-season models for the whole country

Same ablation as `yield_panel_sources.ipynb`, now on all 31 states with rainfed grains (2,211 municipalities). Skill = 1 − RMSE / RMSE(bar); a gain counts when its 95% interval stays above zero.

In [ ]:
def best_table(path, index="features"):
    """Best of fe/gbm skill per feature set, crops and targets as columns."""
    r = pd.read_csv(path)
    index = [index] if isinstance(index, str) else list(index)
    best = r.loc[r.groupby(["crop", "target", *index])["skill"].idxmax()]
    return best.pivot_table(index=index, columns=["crop", "target"], values="skill")

print("Municipality level, 2014-2025")
best_table(NAT / "feature_sets.csv").round(3)

In [ ]:
print("District level (SIAP's unit), 2014-2025")
best_table(NAT / "district_level.csv").round(3)

In [ ]:
nat_r = pd.concat([pd.read_csv(NAT / "feature_sets.csv").assign(level="municipal"),
                   pd.read_csv(NAT / "district_level.csv").assign(level="district")])
nat_r[nat_r.gain_ci_low > 0].sort_values("skill", ascending=False)[["level", "crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high"]].round(3)

In [ ]:
print("Same weather cell and year: municipalities in the same vs another SIAP district")
pd.read_csv(NAT / "district_check.csv").round(3)

### More years at district and state level (in-season)

POWER season weather (plus CHIRPS) on the long panels: districts tested 2010-2025, states 2010-2025, with training back to 1999 and 1982.

In [ ]:
best_table(OUT / "results_inseason_long.csv", index=["level", "features"]).round(3)

## 3. The planning model: what is known by April 30

| Group | Features |
|---|---|
| ocean | ONI Dec-Feb and Feb-Apr, ONI tendency, last season's Jul-Sep ONI, Nino 3.4 (April), tropical North Atlantic and PDO (Feb-Apr) |
| ocean x place | the ocean indices times standardized latitude, longitude, altitude and typical season rain (teleconnections differ across Mexico) |
| preseason (local) | Nov-Apr rain, Mar-Apr rain, Mar-Apr Tmax, Tmin and humidity (POWER) |
| last season weather | last season's rain totals, canicula rain, late rain, dry spell, onset |
| last season outcome | last season's yield and failure rate (partly SIAP reporting persistence) |

Test years: municipalities 2014-2025, districts 2010-2025, states 1995-2025 (31 test years).

In [ ]:
plan = pd.read_csv(OUT / "results_planning.csv")
order = list(planning.planning_sets(pd.read_parquet(OUT / "district.parquet").columns, with_ndvi=False))
for level in ["state", "district", "municipal"]:
    r = plan[plan.level == level]
    best = r.loc[r.groupby(["crop", "target", "features"])["skill"].idxmax()]
    print(f"\n{level}"); display(best.pivot_table(index="features", columns=["crop", "target"], values="skill").reindex(order).round(3))

In [ ]:
print("Planning feature sets whose gain over the bar has a 95% interval above zero:")
plan[plan.gain_ci_low > 0].sort_values(["level", "skill"], ascending=[True, False])[["level", "crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high", "years"]].round(3)

In [ ]:
print("Pre-season NDVI (Mar-Apr), municipalities 2014-2025:")
pd.read_csv(OUT / "results_planning_ndvi.csv")[["crop", "target", "features", "model", "skill", "gain_ci_low", "gain_ci_high"]].round(3)

### What the ocean model learned (state level, maize failure)

In [ ]:
from src.yield_panel import models as pm
st = pd.read_parquet(OUT / "state.parquet")
d = st[st.crop == "maize"].dropna(subset=["failure_rate", *planning.OCEAN])
fit = pm.FixedEffects(["year", *planning.OCEAN], alpha=float(len(planning.OCEAN))).fit(d, "failure_rate")
sd = (d[planning.OCEAN] - d.groupby(pm.UNIT)[planning.OCEAN].transform("mean")).std()
(fit.coef_.drop("year") * sd).rename("change in failure rate per 1 SD").round(4).to_frame()

## 4. A 2026 outlook, as it would have been issued on April 30, 2026

For each district and crop the outlook uses the pre-registered choice (`plan_build.best_set`): the most skilful planning set whose gain interval stayed above zero in testing, and **the bar itself when nothing beat it**. The 10-90% ranges come from each model's own out-of-sample errors. SIAP's 2026 figures are not published yet, so this cannot be scored.

In [ ]:
look = pd.read_csv(OUT / "outlook_2026_district.csv")
look.groupby(["crop", "target"])[["features", "model", "significant"]].first()

In [ ]:
def weighted(f, c):
    return (f[c] * f.sown_ha_last).sum() / f.sown_ha_last.sum()

rows = []
for crop in ["maize", "bean", "sorghum"]:
    f = look[(look.crop == crop) & (look.target == "yield_t_ha")]
    rows.append({"crop": crop, "districts": len(f), "model_median": weighted(f, "model_prediction"),
                 "model_p10": weighted(f, "model_p10"), "model_p90": weighted(f, "model_p90"),
                 "bar_median": weighted(f, "bar_prediction"), "bar_p10": weighted(f, "bar_p10"), "bar_p90": weighted(f, "bar_p90")})
print("2026 yield outlook (t/ha, sown-area-weighted across districts)")
pd.DataFrame(rows).set_index("crop").round(2)

In [ ]:
dist = pd.read_parquet(OUT / "district.parquet")[["state_id", "mun_id", "lat", "lon"]].drop_duplicates(["state_id", "mun_id"])
y = look[(look.crop == "maize") & (look.target == "yield_t_ha")].merge(dist, on=["state_id", "mun_id"])
y["vs_usual_pct"] = 100 * (y.model_prediction / y.bar_prediction - 1)
f = look[(look.crop == "maize") & (look.target == "failure_rate")].merge(dist, on=["state_id", "mun_id"])
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
size = lambda t: np.clip(t.sown_ha_last / 1500, 8, 200)
a = axes[0].scatter(y.lon, y.lat, c=y.vs_usual_pct, s=size(y), cmap="RdBu", vmin=-25, vmax=25, edgecolor="k", linewidth=0.3)
plt.colorbar(a, ax=axes[0], label="2026 expected yield vs the district's average + trend (%)")
axes[0].set(title="Rainfed maize yield outlook 2026 (last season's yield model)", xlabel="lon", ylabel="lat")
b = axes[1].scatter(f.lon, f.lat, c=f["model_p_above_0.3"], s=size(f), cmap="Reds", vmin=0, vmax=0.5, edgecolor="k", linewidth=0.3)
plt.colorbar(b, ax=axes[1], label="P(losing >30% of the sown area), from the district's own history")
axes[1].set(title="Rainfed maize failure risk 2026 (no pre-season signal beat history)", xlabel="lon", ylabel="lat")
fig.tight_layout(); plt.show()

## 5. Notes

* Seasonal climate forecasts (NASA GEOS-S2S, in the NMME archive at the IRI Data Library) would be the natural next planning input; the IRI library now needs a free account, so they are not included here.
* Lagged outcomes partly capture SIAP's own persistence (district estimates repeated year to year), so sets without them are the cleaner test of predictability.
* The shared data package (`python -m src.yield_panel.share export`) holds every processed table used here, so teammates can start from section 1 without downloading anything.